In [0]:
dbutils.widgets.text("catalog", "claims")
catalog = dbutils.widgets.get("catalog")

def n(sql): return spark.sql(sql).collect()[0][0]

clean = f"{catalog}.silver.claims_clean"
quar  = f"{catalog}.silver.claims_quarantine"
brz   = f"{catalog}.bronze.claims"
FROZEN = "DATE'2026-06-30'"

# --- one check per rule: count of clean rows that VIOLATE the rule (must be 0) ---
rule_violations = {
  "passes_q_a": n(f"""
    SELECT COUNT(*) FROM {clean} c
    LEFT JOIN {catalog}.raw.hospitals h ON c.hospital_id = h.hospital_id
    LEFT JOIN {catalog}.raw.members   m ON c.member_id   = m.member_id
    WHERE h.hospital_id IS NULL OR m.member_id IS NULL"""),
  "passes_q_b": n(f"""
    SELECT COUNT(*) FROM {clean}
    WHERE to_date(discharge_date) < to_date(admission_date)
       OR to_date(admission_date) > {FROZEN}
       OR to_date(discharge_date) > {FROZEN}"""),
  "passes_q_c": n(f"""
    SELECT COUNT(*) FROM {clean}
    WHERE CAST(billed_amount AS DOUBLE) <= 0
       OR CAST(approved_amount AS DOUBLE) > CAST(billed_amount AS DOUBLE)"""),
  "passes_q_d": n(f"""
    SELECT COUNT(*) FROM {clean}
    WHERE status <> CASE
        WHEN rejection_code IS NOT NULL THEN 'rejected'
        WHEN settled_ts     IS NOT NULL THEN 'settled'
        WHEN approved_amount IS NOT NULL THEN 'approved'
        ELSE 'submitted' END"""),
  "passes_q_e": n(f"""
    SELECT COUNT(*) FROM {clean}
    WHERE CAST(room_days AS INT) <> datediff(to_date(discharge_date), to_date(admission_date))"""),
}

# --- structural checks ---
n_clean = n(f"SELECT COUNT(*) FROM {clean}")
n_quar  = n(f"SELECT COUNT(*) FROM {quar}")
n_brz   = n(f"SELECT COUNT(DISTINCT claim_id) FROM {brz}")

structural = {
  "clean_not_empty":  n_clean == 0,
  "clean_unique_claim_id": n(f"SELECT COUNT(*) FROM (SELECT claim_id FROM {clean} GROUP BY claim_id HAVING COUNT(*) > 1)") > 0,
  "no_clean_quarantine_overlap": n(f"SELECT COUNT(*) FROM {clean} c JOIN {quar} q ON c.claim_id = q.claim_id") > 0,
  "no_claim_lost": n_clean + n_quar != n_brz,
}

checks = {**{k: v > 0 for k, v in rule_violations.items()}, **structural}
dq_problems = int(sum(checks.values()))

# --- report ---
print(f"clean={n_clean} quarantine={n_quar} distinct_bronze={n_brz}")
for k, v in rule_violations.items():
    print(f"{'FAIL' if v else 'ok  '} {k}: {v} clean rows violate")
for k, failed in structural.items():
    print(f"{'FAIL' if failed else 'ok  '} {k}")
print("quarantine by rule (info only):")
spark.sql(f"SELECT r AS rule, COUNT(*) AS claims FROM {quar} LATERAL VIEW explode(failed_rules) t AS r GROUP BY r ORDER BY r").show()
print("dq_problems =", dq_problems)

dbutils.jobs.taskValues.set(key="dq_problems", value=dq_problems)